# Initial Setups

## Imports + Libraries

In [130]:
import numpy as np
import pandas as pd
import matplotlib as plt
import gutenbergpy
import os
import json
import time
import requests


from gutenbergpy.textget import get_text_by_id, strip_headers
from gutenbergpy.gutenbergcache import GutenbergCache, GutenbergCacheSettings

In [14]:
data_dir = os.path.abspath(os.path.join("..", "data", "gutenberg_catalog_cache"))

GutenbergCacheSettings.set(
    CacheFilename=os.path.join(data_dir, "gutenbergindex.db"),
    CacheUnpackDir=os.path.join(data_dir, "epub"),
)


print(GutenbergCacheSettings.CACHE_FILENAME)
print(GutenbergCacheSettings.CACHE_RDF_UNPACK_DIRECTORY)


/Users/ethanleitch/Documents/Coding_Projects/Socrates_AI/data/gutenberg_catalog_cache/gutenbergindex.db
/Users/ethanleitch/Documents/Coding_Projects/Socrates_AI/data/gutenberg_catalog_cache/epub


In [17]:
GutenbergCache.create(
    refresh=False,
    download=False,
    unpack=False,
    parse=True,
    cache=True,
    deleteTemp=True,
)

 Processing progress: 78964 / 78965 : [###################]]RDF PARSING took 463.78524708747864
 SQLite progress : [###################]]sql took 259.611137
Deleting temporary files
Done


## Creating the Gutenberg cache files

In [ ]:
# GutenbergCache.create(
#     refresh=True,
#     download=True,
#     unpack=True,
#     parse=True,
#     cache=True,
#     deleteTemp=True,
# )

Deleting old files
 Extracting  rdf-files.tar.bz2 : [###################]]took 595.015175
RDF PARSING took 0.005160093307495117
sql took 0.027530
Deleting temporary files
Done


# Data

## Sanity checks

In [20]:
db_path = os.path.join("..", "data", "gutenberg_catalog_cache", "gutenbergindex.db")
print(os.path.exists(db_path))
print(os.path.getsize(db_path))

True
343547904


In [22]:
cache = GutenbergCache.get_cache()
print(cache.native_query("SELECT COUNT(*) FROM books").fetchall())

[(78965,)]


In [47]:
tables = cache.native_query("SELECT name FROM sqlite_master WHERE type='table'")
print([t[0] for t in tables])

['types', 'sqlite_sequence', 'titles', 'subjects', 'rights', 'publishers', 'languages', 'downloadlinkstype', 'downloadlinks', 'bookshelves', 'books', 'book_subjects', 'book_authors', 'authors']


In [26]:
cols = cache.native_query("PRAGMA table_info(books)")
for c in cols:
    print(c) 

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'publisherid', 'INTEGER', 0, None, 0)
(2, 'dateissued', 'DATE', 0, None, 0)
(3, 'rightsid', 'INTEGER', 0, None, 0)
(4, 'numdownloads', 'INTEGER', 0, None, 0)
(5, 'languageid', 'INTEGER', 0, None, 0)
(6, 'bookshelveid', 'INTEGER', 0, None, 0)
(7, 'gutenbergbookid', 'INTEGER', 0, None, 0)
(8, 'typeid', 'INTEGER', 0, None, 0)


In [27]:
for t in ["titles", "authors", "book_authors", "subjects", "book_subjects"]:
    cols = cache.native_query(f"PRAGMA table_info({t})")
    print(f"--- {t} ---")
    for c in cols:
        print(c)

--- titles ---
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 0, None, 0)
(2, 'bookid', 'INTEGER', 0, None, 0)
--- authors ---
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 0, None, 0)
--- book_authors ---
(0, 'bookid', 'INTEGER', 0, None, 0)
(1, 'authorid', 'INTEGER', 0, None, 0)
--- subjects ---
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 0, None, 0)
--- book_subjects ---
(0, 'bookid', 'INTEGER', 0, None, 0)
(1, 'subjectid', 'INTEGER', 0, None, 0)


## Creating new datasets

In [49]:
books = pd.DataFrame(
    cache.native_query("SELECT * FROM books").fetchall(),
    columns=[
        "id",
        "publisherid",
        "dateissued",
        "rightsid",
        "numdownloads",
        "languageid",
        "bookshelfid",
        "gutenbergbookid",
        "typeid",
    ],
)

titles = pd.DataFrame(
    cache.native_query("SELECT * FROM titles").fetchall(),
    columns=["id", "name", "bookid"],
)

authors = pd.DataFrame(
    cache.native_query("SELECT * FROM authors").fetchall(),
    columns=["id", "name"],
)

subjects = pd.DataFrame(
    cache.native_query("SELECT * FROM subjects").fetchall(),
    columns=["id", "name"],
)

languages = pd.DataFrame(
    cache.native_query("SELECT * FROM languages").fetchall(),
    columns=["id", "name"],
)

publishers = pd.DataFrame(
    cache.native_query("SELECT * FROM publishers").fetchall(),
    columns=["id", "name"],
)

rights = pd.DataFrame(
    cache.native_query("SELECT * FROM rights").fetchall(),
    columns=["id", "name"],
)

types = pd.DataFrame(
    cache.native_query("SELECT * FROM types").fetchall(),
    columns=["id", "name"],
)

bookshelves = pd.DataFrame(
    cache.native_query("SELECT * FROM bookshelves").fetchall(),
    columns=["id", "name"],
)

downloadlinks = pd.DataFrame(
    cache.native_query("SELECT * FROM downloadlinks").fetchall(),
    columns=["id", "name", "downloadtypeid", "bookid"],
)

downloadlinkstype = pd.DataFrame(
    cache.native_query("SELECT * FROM downloadlinkstype").fetchall(),
    columns=["id", "name"],
)

book_authors = pd.DataFrame(
    cache.native_query("SELECT * FROM book_authors").fetchall(),
    columns=["bookid", "authorid"],
)

book_subjects = pd.DataFrame(
    cache.native_query("SELECT * FROM book_subjects").fetchall(),
    columns=["bookid", "subjectid"],
)

In [51]:
books.head()

,id,publisherid,dateissued,rightsid,numdownloads,languageid,bookshelfid,gutenbergbookid,typeid
0,1,1,2005-06-02,1,340,1,1,15970,1
1,2,1,2012-04-06,1,223,2,5,39386,1
2,3,1,2022-07-15,1,398,1,7,68524,1
3,4,1,2011-12-09,1,360,1,2,38254,1
4,5,1,2006-05-03,1,1045,1,10,1069,1


In [112]:
titles[titles.bookid == 61804]

,id,name,bookid
65995,65996,Ethics,61804
65996,65997,Ethica. English,61804
65997,65998,Ethica Ordine Geometrico Demonstrata,61804


In [89]:
book_subjects[book_subjects.bookid == 44658]


,bookid,subjectid
150202,44658,701
150203,44658,362
150204,44658,363
150205,44658,364
150206,44658,365
150207,44658,129


In [102]:
subjects[subjects.id == 362]

,id,name
361,362,Political science -- Early works to 1800


In [124]:
books[books.gutenbergbookid == 35722]

,id,publisherid,dateissued,rightsid,numdownloads,languageid,bookshelfid,gutenbergbookid,typeid
8805,8806,1,2011-03-30,1,1153,1,137,35722,1


In [121]:
types

,id,name
0,1,Text
1,2,Sound
2,3,Image
3,4,Dataset
4,5,Collection
5,6,MovingImage
6,7,StillImage


In [88]:
books[books.gutenbergbookid == 1497].id.values[0]

np.int64(44658)

In [127]:
import requests

candidate_urls = [
    "https://www.gutenberg.org/files/35722/35722-0.txt",
    "https://www.gutenberg.org/files/35722/35722.txt",
    "https://www.gutenberg.org/cache/epub/35722/pg35722.txt",
]

for url in candidate_urls:
    resp = requests.get(url)
    print(url, resp.status_code)

https://www.gutenberg.org/files/35722/35722-0.txt 200
https://www.gutenberg.org/files/35722/35722.txt 200
https://www.gutenberg.org/cache/epub/35722/pg35722.txt 200


In [135]:
query = """
SELECT b.gutenbergbookid, b.numdownloads, bs.name, MIN(t.name) AS title
FROM books b
JOIN bookshelves bs ON bs.id = b.bookshelveid
JOIN titles t ON t.bookid = b.id
JOIN types ty ON ty.id = b.typeid
WHERE bs.name = 'Philosophy'
  AND ty.name = 'Text'
GROUP BY b.id
ORDER BY b.numdownloads DESC
LIMIT 50
"""
results = cache.native_query(query).fetchall()
for r in results:
    print(r)

(1998, 25562, 'Philosophy', 'Also sprach Zarathustra. English')
(4363, 20703, 'Philosophy', 'Beyond Good and Evil')
(1497, 17172, 'Philosophy', 'The Republic')
(27942, 14956, 'Philosophy', 'A System of Logic, Ratiocinative and Inductive')
(26495, 11988, 'Philosophy', 'A System of Logic, Ratiocinative and Inductive (Vol. 1 of 2)')
(4280, 10972, 'Philosophy', 'The Critique of Pure Reason')
(34901, 10135, 'Philosophy', 'On Liberty')
(5827, 9852, 'Philosophy', 'The Problems of Philosophy')
(3800, 9340, 'Philosophy', 'Ethica Ordine Geometrico Demonstrata')
(25110, 9287, 'Philosophy', 'The Approach to Philosophy')
(27597, 8195, 'Philosophy', 'The English Utilitarians, Volume 1 (of 3)')
(25447, 8121, 'Philosophy', 'Mysticism and Logic and Other Essays')
(22364, 7858, 'Philosophy', 'The Philosophy of the Moral Feelings')
(852, 7694, 'Philosophy', 'Democracy and Education: An Introduction to the Philosophy of Education')
(25012, 6435, 'Philosophy', 'The Case of Wagner, Nietzsche Contra Wagner, 

In [136]:
top_50_philosophy_ids = [r[0] for r in results]
print(top_50_philosophy_ids)

[1998, 4363, 1497, 27942, 26495, 4280, 34901, 5827, 3800, 25110, 27597, 25447, 22364, 852, 25012, 4705, 19322, 7205, 37090, 11224, 22283, 4320, 1726, 1642, 4352, 23422, 6798, 1635, 2529, 28696, 1016, 989, 4723, 11100, 25172, 35722, 690, 32547, 4763, 11984, 5717, 4391, 990, 7514, 31796, 992, 6366, 12004, 32168, 17556]


In [137]:

def fetch_book_text(book_id):
    """
    Custom fetch funciton for when get_text_by_id() fails
    """
    urls_to_try = [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
    ]
    for url in urls_to_try:
        resp = requests.get(url)
        if resp.status_code == 200:
            raw_bytes = resp.content
            try:
                return raw_bytes.decode("utf-8")
            except UnicodeDecodeError:
                return raw_bytes.decode("latin-1")  # latin-1 never fails, maps every byte 0-255
    raise ValueError(f"Could not fetch text for {book_id}")

text = fetch_book_text(35722)
print(len(text))

1302171


In [139]:
output_dir = os.path.join("..", "data", "raw_books")
os.makedirs(output_dir, exist_ok=True)

metadata_path = os.path.join("..", "data", "metadata.json")
metadata = {}

# --- Step 1: gather metadata + confirm bookshelf ---
for book_id in top_50_philosophy_ids:
    title_query = f"""
    SELECT t.name, bs.name
    FROM books b
    JOIN titles t ON t.bookid = b.id
    JOIN bookshelves bs ON bs.id = b.bookshelveid
    WHERE b.gutenbergbookid = {book_id}
    """
    results = cache.native_query(title_query).fetchall()
    title = results[0][0] if results else "UNKNOWN"
    bookshelf = results[0][1] if results else "UNKNOWN"

    author_query = f"""
    SELECT a.name
    FROM books b
    JOIN book_authors ba ON ba.bookid = b.id
    JOIN authors a ON a.id = ba.authorid
    WHERE b.gutenbergbookid = {book_id}
    """
    author_results = cache.native_query(author_query).fetchall()
    authors = [r[0] for r in author_results]

    is_philosophy = bookshelf == "Philosophy"
    flag = "✓" if is_philosophy else "✗ CHECK THIS ONE"

    print(f"{book_id}: {title} — {authors} — {bookshelf} {flag}")

    metadata[book_id] = {
        "title": title,
        "authors": authors,
        "bookshelf": bookshelf,
        "is_flagged_philosophy": is_philosophy,
        "filename": f"{book_id}.txt",
    }

# --- Step 2: download texts ---
for book_id in top_50_philosophy_ids:
    filepath = os.path.join(output_dir, f"{book_id}.txt")

    if os.path.exists(filepath):
        print(f"Skipping {book_id}, already downloaded")
        metadata[book_id]["char_count"] = os.path.getsize(filepath)
        continue

    try:
        try:
            raw = get_text_by_id(book_id)
            text = strip_headers(raw).decode("utf-8", errors="ignore")
        except UnicodeDecodeError as e:
            print(f"gutenbergpy encoding failed on {book_id} ({e}), trying direct download...")
            text = fetch_book_text(book_id)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(text)

        metadata[book_id]["char_count"] = len(text)
        print(f"Saved {book_id} ({len(text)} chars)")

    except Exception as e:
        print(f"Failed on {book_id}: {e}")
        metadata[book_id]["char_count"] = None
        metadata[book_id]["error"] = str(e)

    time.sleep(1)

# --- Step 3: save metadata ---
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\nMetadata saved to {metadata_path}")
print(f"Texts saved to {output_dir}")

1998: Thus Spake Zarathustra: A Book for All and None — ['Nietzsche, Friedrich Wilhelm'] — Philosophy ✓
4363: Beyond Good and Evil — ['Nietzsche, Friedrich Wilhelm'] — Philosophy ✓
1497: The Republic — ['Πλάτων', 'Plato'] — Philosophy ✓
27942: A System of Logic, Ratiocinative and Inductive — ['Mill, J. S. (John Stuart)', 'Mill, John Stuart'] — Philosophy ✓
26495: A System of Logic, Ratiocinative and Inductive (Vol. 1 of 2) — ['Mill, J. S. (John Stuart)', 'Mill, John Stuart'] — Philosophy ✓
4280: The Critique of Pure Reason — ['Kant, Emmanuel', 'Kant, I. (Immanuil)', 'Kant, Immanuel'] — Philosophy ✓
34901: On Liberty — ['Mill, J. S. (John Stuart)', 'Mill, John Stuart'] — Philosophy ✓
5827: The Problems of Philosophy — ['Russell, Bertrand Russell, 3d Earl', 'Russell, Bertrand A. W. (Bertrand Arthur William)', 'Russell, Bertrand'] — Philosophy ✓
3800: Ethics — ['Spinoza, Benedict of', 'Spinoza, Baruch', 'Espinosa, Baruch de', 'De Spinoza, Benedictus', 'Spinoza, Benedictus de'] — Philosoph

In [24]:
test_id = 1497  # Republic, one you know for certain exists

result = cache.native_query(f"SELECT * FROM books WHERE gutenbergbookid = {test_id}").fetchall()
print(result)

[(44658, 1, '1998-10-01', 1, 17172, 1, 137, 1497, 1)]


In [ ]:
ALPHABET = list("abcdefghijklmnopqrstuvwxyz .,!?;:'\"()-\n")
VOCAB = {ch:i for i,ch in enumerate(ALPHABET)} # returns an integer given a character
INV_VOCAB = {i:ch for ch,i in VOCAB.items()} # returns a character given an integer
L = len(ALPHABET)